# Phase 2 — YOLOv8n ROI Detector: Dataset Preparation + Training + Evaluation

**Pipeline:**
1. Generate ROI bounding boxes from segmentation masks
2. Build YOLO-format dataset (train/val/test split)
3. Train YOLOv8n ROI detector
4. Evaluate detector (mAP, precision, recall, F1, FPS, energy)
5. Export all metrics to CSV


In [ ]:
# ── 1. Mount Drive ──────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── 2. Clone project from GitHub ────────────────────────────────────────────
import os, shutil

GITHUB_REPO   = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'
PROJECT_LOCAL = '/content/your_project_name'   # ← update to match your repo

if os.path.exists(PROJECT_LOCAL):
    shutil.rmtree(PROJECT_LOCAL)

!git clone {GITHUB_REPO} {PROJECT_LOCAL}
os.chdir(PROJECT_LOCAL)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# ── 3. Install dependencies ──────────────────────────────────────────────────
!python install_dependencies.py

In [ ]:
# ── 4. Setup ─────────────────────────────────────────────────────────────────
import sys, logging
sys.path.insert(0, os.getcwd())

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)

import torch
print(f'PyTorch {torch.__version__} | GPU: {torch.cuda.is_available()}')

In [ ]:
# ── 5. Load detector config ──────────────────────────────────────────────────
from utils.config_loader import load_config

cfg = load_config('configs/detector_config.yaml')
print(f'Project : {cfg.project.name}')
print(f'Lesion class IDs: {cfg.bbox.lesion_class_ids}')
print(f'Device  : {cfg.training.device}')

## Step A — ROI Dataset Preparation

In [ ]:
# ── 6. Generate bboxes + build YOLO dataset ──────────────────────────────────
from detectors.prepare_dataset import run_dataset_preparation

prep_result = run_dataset_preparation(cfg)
bbox_results = prep_result['bbox_results']
splits       = prep_result['splits']

In [ ]:
# ── 7. Visualise generated bboxes ────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

img = mpimg.imread('results/detector/visualizations/bbox_samples.png')
plt.figure(figsize=(16, 9))
plt.imshow(img)
plt.axis('off')
plt.title('Generated ROI Bounding Boxes — Sample Preview')
plt.show()

In [ ]:
# ── 8. Inspect individual sample (mask vs bbox) ──────────────────────────────
import cv2, random
from detectors.roi_visualizer import visualize_mask_vs_bbox

sample_file = random.choice(list(bbox_results.keys()))
img      = cv2.imread(cfg.data.image_dir + '/' + sample_file, cv2.IMREAD_GRAYSCALE)
mask_bgr = cv2.imread(cfg.data.mask_dir  + '/' + sample_file, cv2.IMREAD_COLOR)
boxes    = bbox_results[sample_file]

visualize_mask_vs_bbox(
    image    = img,
    mask_bgr = mask_bgr,
    boxes    = boxes,
    save_path= 'results/detector/visualizations/single_sample_bbox.png',
    title    = sample_file,
)

img2 = mpimg.imread('results/detector/visualizations/single_sample_bbox.png')
plt.figure(figsize=(14, 4)); plt.imshow(img2); plt.axis('off'); plt.show()

## Step B — YOLOv8n Training

In [ ]:
# ── 9. Train detector ────────────────────────────────────────────────────────
# Training logs + TensorBoard data go to:
#   experiments/detector_runs/yolov8n_oct_roi/

from training.train_detector import train_detector

best_pt = train_detector(cfg)
print(f'\nBest checkpoint: {best_pt}')

In [ ]:
# ── 10. (Optional) Launch TensorBoard ───────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir experiments/detector_runs

## Step C — Evaluation & Profiling

In [ ]:
# ── 11. Full evaluation ──────────────────────────────────────────────────────
from evaluation.evaluate_detector import run_detector_evaluation

bundle = run_detector_evaluation(cfg, experiment_name='yolov8n_baseline')

In [ ]:
# ── 12. Profiling (FLOPs / params / memory / latency) ───────────────────────
from evaluation.profile_detector import profile_yolo_detector, save_profile_csv

profile = profile_yolo_detector(
    checkpoint_path   = cfg.detector.checkpoint,
    input_shape       = tuple(cfg.profiling.flops_input_shape),
    device            = cfg.detector.device,
    num_latency_runs  = 100,
)
save_profile_csv(profile, 'results/detector/profiling/detector_profile.csv')

In [ ]:
# ── 13. View metrics CSV ─────────────────────────────────────────────────────
import pandas as pd
df = pd.read_csv(cfg.evaluation.metrics_csv)
df.T

## Step D — Inference Demo

In [ ]:
# ── 14. Single image inference demo ─────────────────────────────────────────
import cv2, matplotlib.pyplot as plt
from inference.roi_inference    import load_detector
from inference.roi_cropper      import crop_roi
from detectors.roi_visualizer   import draw_yolo_detections

detector = load_detector(cfg)

# Pick any OCT image
demo_path = cfg.data.image_dir + '/' + list(bbox_results.keys())[0]
img       = cv2.imread(demo_path, cv2.IMREAD_GRAYSCALE)

detections = detector.predict(img, apply_padding=True)
print(f'Detections: {len(detections)}')
for d in detections:
    print(f'  [{d.x1},{d.y1},{d.x2},{d.y2}]  conf={d.conf:.3f}')

# Visualise
vis = draw_yolo_detections(img, [d.to_dict() for d in detections])

# Crop ROI
if detections:
    best = detections[0]
    crop, coords = crop_roi(img, best.x1, best.y1, best.x2, best.y2,
                             target_size=256, keep_aspect_ratio=True)

fig, axes = plt.subplots(1, 3 if detections else 2, figsize=(14, 4))
axes[0].imshow(img, cmap='gray');  axes[0].set_title('OCT Input')
axes[1].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); axes[1].set_title('Detection')
if detections:
    axes[2].imshow(crop, cmap='gray'); axes[2].set_title('Cropped ROI (→ Phase 3)')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

## Results Location
| Artifact | Path |
|---|---|
| Detector metrics CSV | `results/detector/metrics/detector_metrics.csv` |
| Profiling CSV | `results/detector/profiling/detector_profile.csv` |
| BBox samples | `results/detector/visualizations/bbox_samples.png` |
| Accuracy bar chart | `results/detector/visualizations/detector_accuracy_metrics.png` |
| Training runs | `experiments/detector_runs/yolov8n_oct_roi/` |
| Best checkpoint | `experiments/detector_runs/yolov8n_oct_roi/weights/best.pt` |

## Next: Phase 3
Phase 3 will use `inference/roi_inference.py` and `inference/roi_cropper.py`
(already built and Phase-agnostic) to feed the cropped ROI into the
Phase 1 segmentation model.
